# 第 7 周 - 笔记本 1：数据准备

## 目标
准备用于微调 Llama 3.2 的训练数据：
1. 从 HuggingFace Hub 加载数据集
2. 分析代币分布
3. 创建提示-完成对
4.上传至HuggingFace Hub

## 时间：10-15 分钟

In [ ]:
import sys
sys.path.append('..')

import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

from src.items import Item
from src.config import config

# 加载环境变量
# Load environment variables
load_dotenv()
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

print("✅ Environment loaded")

## 显示配置

In [ ]:
config.display()

## 步骤 1：加载数据集

In [ ]:
print(f"Loading dataset: {config.DATASET_NAME}")
train, val, test = Item.from_hub(config.DATASET_NAME)
items = train + val + test

print(f"\n✅ Loaded:")
print(f"   Training: {len(train):,} items")
print(f"   Validation: {len(val):,} items")
print(f"   Test: {len(test):,} items")
print(f"   Total: {len(items):,} items")

## 步骤 2：分析代币分布

In [ ]:
# 加载分词器
# Load tokenizer
print(f"Loading tokenizer: {config.BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
print("✅ Tokenizer loaded")

In [ ]:
# 计算摘要中的标记
# Count tokens in summaries
print("Counting tokens in product summaries...")
token_counts = [item.count_tokens(tokenizer) for item in tqdm(items)]

avg_tokens = sum(token_counts) / len(token_counts)
max_tokens = max(token_counts)

print(f"\n📊 Token Statistics:")
print(f"   Average: {avg_tokens:.1f} tokens")
print(f"   Maximum: {max_tokens} tokens")

In [ ]:
# 可视化代币分布
# Visualize token distribution
plt.figure(figsize=(15, 6))
plt.title(f"Token Distribution: Avg {avg_tokens:.1f}, Max {max_tokens}")
plt.xlabel('Number of tokens in summary')
plt.ylabel('Count')
plt.hist(token_counts, rwidth=0.7, color="skyblue", bins=range(0, 200, 10))
plt.axvline(config.MAX_TOKENS, color='red', linestyle='--', label=f'Cutoff: {config.MAX_TOKENS}')
plt.legend()
plt.show()

In [ ]:
# 检查有多少项将被截断
# Check how many items will be truncated
truncated = len([count for count in token_counts if count > config.MAX_TOKENS])
truncated_pct = truncated / len(items) * 100

print(f"\n📏 With cutoff of {config.MAX_TOKENS} tokens:")
print(f"   {truncated:,} items will be truncated ({truncated_pct:.1f}%)")
print(f"   {len(items) - truncated:,} items fit within limit ({100-truncated_pct:.1f}%)")

## 步骤 3：创建提示完成对

In [ ]:
# 示例：创建提示之前
# Example: Before creating prompts
print("📝 Example item BEFORE prompt creation:")
print(f"\nTitle: {train[0].title}")
print(f"Price: ${train[0].price:.2f}")
print(f"\nSummary:\n{train[0].summary[:200]}...")

In [ ]:
# 为所有项目创建提示
# Create prompts for all items
print("Creating prompt-completion pairs...")

# 培训和验证：圆形价格
# Training and validation: round prices
for item in tqdm(train + val, desc="Train/Val"):
    item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=True)

# 测试：保持准确的价格
# Test: keep exact prices
for item in tqdm(test, desc="Test"):
    item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=False)

print("✅ Prompts created")

In [ ]:
# 示例：创建提示后
# Example: After creating prompts
print("📝 Example item AFTER prompt creation:")
print(f"\n{'='*60}")
print("PROMPT:")
print(f"{'='*60}")
print(test[0].prompt)
print(f"\n{'='*60}")
print("COMPLETION:")
print(f"{'='*60}")
print(test[0].completion)
print(f"{'='*60}")

In [ ]:
# 分析提示+完成令牌计数
# Analyze prompt+completion token counts
print("Counting tokens in full prompts...")
prompt_token_counts = [item.count_prompt_tokens(tokenizer) for item in tqdm(items)]

avg_prompt_tokens = sum(prompt_token_counts) / len(prompt_token_counts)
max_prompt_tokens = max(prompt_token_counts)

print(f"\n📊 Full Prompt Token Statistics:")
print(f"   Average: {avg_prompt_tokens:.1f} tokens")
print(f"   Maximum: {max_prompt_tokens} tokens")

In [ ]:
# 可视化完整的提示令牌分布
# Visualize full prompt token distribution
plt.figure(figsize=(15, 6))
plt.title(f"Full Prompt Token Distribution: Avg {avg_prompt_tokens:.1f}, Max {max_prompt_tokens}")
plt.xlabel('Number of tokens (prompt + completion)')
plt.ylabel('Count')
plt.hist(prompt_token_counts, rwidth=0.7, color="gold", bins=range(0, 200, 10))
plt.axvline(config.MAX_SEQ_LENGTH, color='red', linestyle='--', label=f'Max Seq Length: {config.MAX_SEQ_LENGTH}')
plt.legend()
plt.show()

## 步骤 4：上传至 HuggingFace Hub

In [ ]:
# 上传提示完成数据集
# Upload prompt-completion dataset
print(f"Uploading to: {config.PROMPTS_DATASET_NAME}")
print("This may take a few minutes...")

Item.push_prompts_to_hub(config.PROMPTS_DATASET_NAME, train, val, test)

print(f"\n✅ Dataset uploaded successfully!")
print(f"\n🔗 View at: https://huggingface.co/datasets/{config.PROMPTS_DATASET_NAME}")

## 概括

✅ 数据准备完成！

**我们做了什么：**
1. 从 HuggingFace Hub 加载数据集
2. 分析代币分布（平均约 60 个代币）
3. 将截止值设置为 110 个标记（截断约 5% 的项目）
4. 创建结构化提示-完成对
5.将训练数据上传到HuggingFace Hub

**下一步：** `02_base_model_test.ipynb` - 在微调之前测试基础 Llama